# 01 — Data Collection
Scrapes all PL match data (xG, xGA, results) for Arsenal, Liverpool, Manchester City, and Manchester United from the Arteta era (2019-20 → present) via Understat.

## Imports

In [2]:
import pandas as pd
import soccerdata as sd
import time
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

print(f"pandas      : {pd.__version__}")
print(f"soccerdata  : {sd.__version__}")
print("\n All imports successful!")

pandas      : 3.0.5
soccerdata  : 1.9.1

 All imports successful!


## Project Paths

In [3]:
NOTEBOOK_DIR = Path.cwd()

PROJECT_ROOT = NOTEBOOK_DIR.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

RAW_DATA_DIR.mkdir(parents = True, exist_ok = True)

#confirmation
print(f"Notebook running from : {NOTEBOOK_DIR}")
print(f"Project root          : {PROJECT_ROOT}")
print(f"Raw data directory    : {RAW_DATA_DIR}")
print(f"Raw dir exists        : {RAW_DATA_DIR.exists()}")

Notebook running from : C:\Users\tejas\OneDrive\Desktop\Arsenal Bottle Python\notebooks
Project root          : C:\Users\tejas\OneDrive\Desktop\Arsenal Bottle Python
Raw data directory    : C:\Users\tejas\OneDrive\Desktop\Arsenal Bottle Python\data\raw
Raw dir exists        : True


## Configuration

In [4]:
#Our 4 title team contendors
titleTeams = ["Arsenal","Manchester City","Liverpool","Manchester United"]

#Our Date Ranges
artetaSeasons = ['1920', '2021', '2122', '2223', '2324', '2425', '2526']
preArtetaSeasons = [2017,2018]

PL="ENG-Premier League"

## Understat Scraper Setup

In [6]:
#Creating the Understat Scraper

print("Setting up Understat scraper.....")

understat = sd.Understat(
    leagues = PL,
    seasons = artetaSeasons
)

#confirmations
print(f"Scraper type   : {type(understat)}")
print(f"Scraper leagues: {understat.leagues}")
print(f"Scraper seasons: {understat.seasons}")
print("\n Understat scraper ready.")
print("   (First data pull will be slow — cached after that)")

Setting up Understat scraper.....


[08/21/26 16:25:11] INFO     Saving cached data to C:\Users\tejas\soccerdata\data\Understat          ]8;id=9451665;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=9451666;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

Scraper type   : <class 'soccerdata.understat.Understat'>
Scraper leagues: ['ENG-Premier League']
Scraper seasons: ['1920', '2021', '2122', '2223', '2324', '2425', '2526']

 Understat scraper ready.
   (First data pull will be slow — cached after that)


## Scrape Schedule

In [7]:
#Pulling the full schedule for all Arteta Seasons

print(" Scraping EPL schedule (2019-20 to 2025-26)...")
print("   First run takes ~1-2 minutes. Subsequent runs are instant (cached).\n")

schedule_raw = understat.read_schedule()

#confirmation
print(f"\nDone!")
print(f"   Raw shape: {schedule_raw.shape}  (rows, columns)")
print(f"   Index names: {schedule_raw.index.names}")

schedule_raw.head()
schedule_raw.info()

 Scraping EPL schedule (2019-20 to 2025-26)...
   First run takes ~1-2 minutes. Subsequent runs are instant (cached).


Done!
   Raw shape: (2660, 17)  (rows, columns)
   Index names: ['league', 'season', 'game']
<class 'pandas.DataFrame'>
MultiIndex: 2660 entries, ('ENG-Premier League', '1920', '2019-08-09 Liverpool-Norwich') to ('ENG-Premier League', '2526', '2026-05-24 West Ham-Leeds')
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   league_id       2660 non-null   string        
 1   season_id       2660 non-null   Int64         
 2   game_id         2660 non-null   Int64         
 3   date            2660 non-null   datetime64[us]
 4   home_team_id    2660 non-null   Int64         
 5   away_team_id    2660 non-null   Int64         
 6   home_team       2660 non-null   string        
 7   away_team       2660 non-null   string        
 8   away_team_code  2660 non-null   string        
 9 

## Reset Index

In [8]:
schedule= schedule_raw.reset_index()
#schedule = schedule[['league','season','game','date','home_team','away_team','home_goals','away_goals','home_xg','away_xg','is_result','url']]

print("After reset_index():")
print(f"  Shape  : {schedule.shape}")
print(f"  Columns: {list(schedule.columns)}")
print()
schedule.head(3)

After reset_index():
  Shape  : (2660, 20)
  Columns: ['league', 'season', 'game', 'league_id', 'season_id', 'game_id', 'date', 'home_team_id', 'away_team_id', 'home_team', 'away_team', 'away_team_code', 'home_team_code', 'home_goals', 'away_goals', 'home_xg', 'away_xg', 'is_result', 'has_data', 'url']



,league,season,game,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,away_team,away_team_code,home_team_code,home_goals,away_goals,home_xg,away_xg,is_result,has_data,url
0,ENG-Premier League,1920,2019-08-09 Liverpool-Norwich,1,2019,11643,2019-08-09 20:00:00,87,79,Liverpool,Norwich,NOR,LIV,4,1,2.23456,0.842407,True,True,https://understat.com/match/11643
1,ENG-Premier League,1920,2019-08-10 Bournemouth-Sheffield United,1,2019,11645,2019-08-10 15:00:00,73,238,Bournemouth,Sheffield United,SHE,BOU,1,1,1.34099,1.59864,True,True,https://understat.com/match/11645
2,ENG-Premier League,1920,2019-08-10 Burnley-Southampton,1,2019,11646,2019-08-10 15:00:00,92,74,Burnley,Southampton,SOU,BUR,3,0,0.909241,1.08752,True,True,https://understat.com/match/11646


## Reshape: Wide Format → Long Format
The schedule already contains `home_xg` and `away_xg` — no second scrape needed.
Split into home/away perspectives, flip xG and xGA for the away team, then stack with `pd.concat()`.

In [9]:
#columns we want to keep
KEEP_COLS = ['league', 'season', 'game_id', 'date', 
             'team', 'opponent', 'venue', 'xG', 'xGA', 'scored', 'conceded']

#Home Team Perspective

home = schedule.rename(columns={
    'home_team' : 'team',
    'away_team' : 'opponent',
    'home_xg' : 'xG',
    'away_xg' : 'xGA',
    'away_goals' : 'scored',
    'home_goals' : 'conceded'
})
home['venue'] = 'home'

#Away Team Perspective

away = schedule.rename(columns={
    'away_team' : 'team',
    'home_team' : 'opponent',
    'away_xg' : 'xG',
    'home_xg' : 'xGA',
    'away_goals' : 'scored',
    'home_goals' : 'conceded'
})
away['venue'] = 'away'

#Stack Vertically

matches=pd.concat(
    [home[KEEP_COLS], away[KEEP_COLS]], ignore_index = True
)


print(f"Schedule (wide)   : {len(schedule):,} rows  (one per match)")
print(f"Matches (long)    : {len(matches):,} rows  (one per team per match)")
print(f"Ratio             : {len(matches) / len(schedule):.0f}x  (should be exactly 2)")
print(f"\nColumns: {list(matches.columns)}")

Schedule (wide)   : 2,660 rows  (one per match)
Matches (long)    : 5,320 rows  (one per team per match)
Ratio             : 2x  (should be exactly 2)

Columns: ['league', 'season', 'game_id', 'date', 'team', 'opponent', 'venue', 'xG', 'xGA', 'scored', 'conceded']


In [10]:
first_id = matches['game_id'].iloc[0]
matches[matches['game_id'] == first_id][['team', 'opponent', 'venue', 'xG', 'xGA', 'scored', 'conceded']]

,team,opponent,venue,xG,xGA,scored,conceded
0,Liverpool,Norwich,home,2.23456,0.842407,1,4
2660,Norwich,Liverpool,away,0.842407,2.23456,1,4


## Explore the Data

In [11]:
all_teams = sorted(matches['team'].unique())
print(f"Total unique teams: {len(all_teams)}\n")
print("All teams:")
for t in all_teams:
    print(f"  '{t}'")

Total unique teams: 28

All teams:
  'Arsenal'
  'Aston Villa'
  'Bournemouth'
  'Brentford'
  'Brighton'
  'Burnley'
  'Chelsea'
  'Crystal Palace'
  'Everton'
  'Fulham'
  'Ipswich'
  'Leeds'
  'Leicester'
  'Liverpool'
  'Luton'
  'Manchester City'
  'Manchester United'
  'Newcastle United'
  'Norwich'
  'Nottingham Forest'
  'Sheffield United'
  'Southampton'
  'Sunderland'
  'Tottenham'
  'Watford'
  'West Bromwich Albion'
  'West Ham'
  'Wolverhampton Wanderers'


In [12]:
print("Checking our team names against Understat's:")
for team in titleTeams:
    found = team in matches['team'].values
    status = "✅ Found" if found else "❌ NOT FOUND — check spelling"
    print(f"  {team}: {status}")

Checking our team names against Understat's:
  Arsenal: ✅ Found
  Manchester City: ✅ Found
  Liverpool: ✅ Found
  Manchester United: ✅ Found


In [13]:
print("Row count per season (should be ~760 = 380 matches × 2 teams):")
print(matches['season'].value_counts().sort_index())

Row count per season (should be ~760 = 380 matches × 2 teams):
season
1920    760
2021    760
2122    760
2223    760
2324    760
2425    760
2526    760
Name: count, dtype: int64


In [14]:
print("Missing values in key columns:")
key_cols = ['xG', 'xGA', 'scored', 'conceded', 'date', 'team', 'season']
print(matches[key_cols].isna().sum())

Missing values in key columns:
xG          0
xGA         0
scored      0
conceded    0
date        0
team        0
season      0
dtype: int64


## Filter to 4 Title Teams

In [15]:
matches_4teams = matches[matches['team'].isin(titleTeams)].copy()

print(f"All 20 teams (unfiltered) : {len(matches):,} rows")
print(f"Our 4 teams only          : {len(matches_4teams):,} rows")
print()

print("Matches per team:")
print(matches_4teams['team'].value_counts())

All 20 teams (unfiltered) : 5,320 rows
Our 4 teams only          : 1,064 rows

Matches per team:
team
Liverpool            266
Manchester United    266
Arsenal              266
Manchester City      266
Name: count, dtype: Int64


In [16]:
arsenal = matches_4teams[matches_4teams['team'] == 'Arsenal']
print(f"Arsenal rows: {len(arsenal)}")
arsenal[['date', 'opponent', 'venue', 'xG', 'xGA', 'scored', 'conceded']].head(5)

Arsenal rows: 266


,date,opponent,venue,xG,xGA,scored,conceded
10,2019-08-17 12:30:00,Burnley,home,1.1644,1.39172,1,2
38,2019-09-01 16:30:00,Tottenham,home,1.92509,1.95514,2,2
56,2019-09-22 15:30:00,Aston Villa,home,2.53209,1.71546,2,3
76,2019-10-06 14:00:00,Bournemouth,home,1.24109,0.683967,0,1
96,2019-10-27 16:30:00,Crystal Palace,home,1.54014,1.77892,2,2


## Pre-Arteta Arsenal (2017-19)

## 2026-27 Fixture List

In [19]:
#scraing arsenal pre arteta (only 2017-18,18-19)

understat_pre = sd.Understat(leagues=PL, seasons= preArtetaSeasons)
pre_schedule_raw = understat_pre.read_schedule().reset_index()

pre_home = pre_schedule_raw.rename(columns={
    'home_team': 'team', 'away_team': 'opponent',
    'home_xg': 'xG', 'away_xg': 'xGA',
    'home_goals': 'scored', 'away_goals': 'conceded'
})
pre_home['venue'] = 'home'

pre_away = pre_schedule_raw.rename(columns={
    'away_team': 'team', 'home_team': 'opponent',
    'away_xg': 'xG', 'home_xg': 'xGA',
    'away_goals': 'scored', 'home_goals': 'conceded'
})
pre_away['venue'] = 'away'

pre_matches = pd.concat([pre_home[KEEP_COLS],pre_away[KEEP_COLS]], ignore_index = True)

pre_arteta = pre_matches[pre_matches['team'] == 'Arsenal'].copy()

print(f"✅ Arsenal pre-Arteta: {len(pre_arteta)} matches")
print(f"   Seasons: {sorted(pre_arteta['season'].unique())}")
pre_arteta.head(3)

[08/21/26 17:06:38] INFO     Saving cached data to C:\Users\tejas\soccerdata\data\Understat          ]8;id=9451671;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=9451672;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

✅ Arsenal pre-Arteta: 76 matches
   Seasons: ['1718', '1819']


,league,season,game_id,date,team,opponent,venue,xG,xGA,scored,conceded
0,ENG-Premier League,1718,7119,2017-08-11 19:45:00,Arsenal,Leicester,home,2.54329,1.46495,4,3
30,ENG-Premier League,1718,7152,2017-09-09 15:00:00,Arsenal,Bournemouth,home,2.48732,0.577291,3,0
59,ENG-Premier League,1718,7178,2017-09-25 20:00:00,Arsenal,West Bromwich Albion,home,2.39316,0.766652,2,0


## All 20 Teams — 2026-27 (Comparison Feature)

In [27]:
#scraping 2026-27 Schedule ...

understat_2627 = sd.Understat(leagues=PL, seasons=2026)
fixtures_raw   = understat_2627.read_schedule().reset_index()


fixtures_raw

[08/21/26 17:19:27] INFO     Saving cached data to C:\Users\tejas\soccerdata\data\Understat          ]8;id=9451719;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=9451720;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

,index
0,league
1,season
2,game


## Save to CSV

In [29]:
#saving arteta era match data

for team in titleTeams:
    team_df = matches_4teams[matches_4teams['team'] == team].copy()

    if len(team_df) == 0:
        print(f"   ⚠️  No data for {team}")
        continue

    filename = f"{team.lower().replace(' ', '_')}_raw.csv"
    team_df.to_csv(RAW_DATA_DIR / filename, index=False)
    print(f"   ✅ {team}: {len(team_df):,} matches → {filename}")

   ✅ Arsenal: 266 matches → arsenal_raw.csv
   ✅ Manchester City: 266 matches → manchester_city_raw.csv
   ✅ Liverpool: 266 matches → liverpool_raw.csv
   ✅ Manchester United: 266 matches → manchester_united_raw.csv


In [30]:
pre_arteta.to_csv(RAW_DATA_DIR / "arsenal_pre_arteta_raw.csv", index=False)
print(f"✅ Arsenal pre-Arteta   : {len(pre_arteta)} rows")

✅ Arsenal pre-Arteta   : 76 rows


## Sanity Checks

In [31]:
print("=" * 65)
print("SANITY CHECK 1 — Match count per team per season")
print("Expected: 38 per team per full season")
print("=" * 65)

counts = (
    matches_4teams
    .groupby(['team', 'season'])
    .size()
    .unstack(fill_value=0)
)
print(counts)

SANITY CHECK 1 — Match count per team per season
Expected: 38 per team per full season
season             1920  2021  2122  2223  2324  2425  2526
team                                                       
Arsenal              38    38    38    38    38    38    38
Liverpool            38    38    38    38    38    38    38
Manchester City      38    38    38    38    38    38    38
Manchester United    38    38    38    38    38    38    38


In [32]:
print("=" * 65)
print("SANITY CHECK 2 — Missing xG values")
print("=" * 65)

for team in titleTeams:
    team_df  = matches_4teams[matches_4teams['team'] == team]
    missing  = team_df['xG'].isna().sum()
    status   = "✅ None" if missing == 0 else f"⚠️  {missing} missing"
    print(f"  {team}: {status}")

SANITY CHECK 2 — Missing xG values
  Arsenal: ✅ None
  Manchester City: ✅ None
  Liverpool: ✅ None
  Manchester United: ✅ None


In [33]:
print("=" * 65)
print("SANITY CHECK 3 — Date range")
print("=" * 65)

# pd.to_datetime() = as.Date() in R
matches_4teams['date'] = pd.to_datetime(matches_4teams['date'])

print(f"Earliest : {matches_4teams['date'].min().date()}")
print(f"Latest   : {matches_4teams['date'].max().date()}")
print(f"Expected : 2019-08-xx → 2026-05-xx")

SANITY CHECK 3 — Date range
Earliest : 2019-08-09
Latest   : 2026-05-24
Expected : 2019-08-xx → 2026-05-xx


In [35]:
print("=" * 65)
print("SANITY CHECK 4 — xG summary (spot obvious outliers)")
print("Expected xG range: roughly 0.2 – 3.5 per match")
print("=" * 65)

print(matches_4teams[['xG', 'xGA', 'scored', 'conceded']].describe().round(2))

SANITY CHECK 4 — xG summary (spot obvious outliers)
Expected xG range: roughly 0.2 – 3.5 per match
           xG     xGA  scored  conceded
count  1064.0  1064.0  1064.0    1064.0
mean     1.98    1.18    1.36      1.68
std      1.01    0.83    1.23      1.46
min      0.11    0.02     0.0       0.0
25%      1.24    0.56     0.0       1.0
50%      1.86     1.0     1.0       1.0
75%      2.59    1.64     2.0       3.0
max      6.67    5.41     7.0       9.0
